# 04 — Human Likert evaluation power analysis

Use this notebook for a simplified human evaluation setup:

- each item is rated for Model A and Model B;
- several raters may rate each item;
- ratings are normalized to a 0–1 scale.

This is a simplified classroom version. For serious analysis, a mixed-effects model is often more appropriate, especially when the same annotators rate multiple items.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from power_utils import *


## Student input

Enter assumptions about the evaluation design.

Definitions:

- `n_items`: number of generated outputs/items rated.
- `raters_per_item`: number of ratings per item.
- `expected_effect`: expected improvement of Model B over A on a 0–1 scale.
- `item_sd`: how much item quality varies.
- `rater_sd`: how much raters differ in overall strictness.
- `residual_sd`: remaining noise in individual ratings.


In [ ]:
n_items = ask_int("Number of evaluated items", 100, minimum=10)
raters_per_item = ask_int("Raters per item", 3, minimum=1)
baseline_mean = ask_float("Expected mean rating of Model A", 0.60, minimum=0.0, maximum=1.0)
expected_effect = ask_float("Expected improvement of Model B over A", 0.05, minimum=0.0, maximum=1.0)
item_sd = ask_float("Item-level standard deviation", 0.15, minimum=0.0)
rater_sd = ask_float("Rater-bias standard deviation", 0.10, minimum=0.0)
residual_sd = ask_float("Residual rating noise standard deviation", 0.20, minimum=0.0001)
alpha = ask_float("Significance threshold alpha", 0.05, minimum=0.0001, maximum=0.5)
target_power = ask_float("Target power", 0.80, minimum=0.01, maximum=0.99)
n_trials = ask_int("Number of simulation trials", 2000, minimum=100)

params = dict(n_items=n_items, raters_per_item=raters_per_item, baseline_mean=baseline_mean,
              expected_effect=expected_effect, item_sd=item_sd, rater_sd=rater_sd,
              residual_sd=residual_sd, alpha=alpha, target_power=target_power, n_trials=n_trials)
params

## Estimate power for this human evaluation design

In [ ]:
power = estimate_likert_power(
    n_items=n_items,
    raters_per_item=raters_per_item,
    baseline_mean=baseline_mean,
    expected_effect=expected_effect,
    item_sd=item_sd,
    rater_sd=rater_sd,
    residual_sd=residual_sd,
    alpha=alpha,
    n_trials=n_trials,
)
print(f"Estimated power: {power:.3f}")

## Compare designs: more items vs. more raters

This cell helps students see that increasing the number of items and increasing the number of raters are not always equivalent.


In [ ]:
designs = []
for items in [50, 100, 200, 500]:
    for raters in [1, 3, 5, 10]:
        pwr = estimate_likert_power(
            n_items=items,
            raters_per_item=raters,
            baseline_mean=baseline_mean,
            expected_effect=expected_effect,
            item_sd=item_sd,
            rater_sd=rater_sd,
            residual_sd=residual_sd,
            alpha=alpha,
            n_trials=max(500, n_trials // 2),
        )
        designs.append({"n_items": items, "raters_per_item": raters, "power": pwr})

design_df = pd.DataFrame(designs)
design_df

In [ ]:
plt.figure(figsize=(7, 4))
for raters, group in design_df.groupby("raters_per_item"):
    plt.plot(group["n_items"], group["power"], marker="o", label=f"{raters} raters/item")
plt.axhline(target_power, linestyle="--")
plt.xscale("log")
plt.xlabel("Number of items")
plt.ylabel("Estimated power")
plt.title("Human evaluation power by design")
plt.legend()
plt.show()

## Look at the included human-evaluation summary data

This small dataset is included for classroom discussion. It shows reported item counts and effect sizes from a collection of human evaluations.


In [ ]:
human = pd.read_csv(DATA_DIR / "human_eval" / "num_items_vs_effect_size.csv")
human[["Examples", "Ann.Per.Ex", "baseline", "new", "diff", "Metric", "BigCategory"]].head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(human["Examples"], human["diff"].abs(), alpha=0.7)
plt.xscale("log")
plt.xlabel("Number of evaluated items")
plt.ylabel("Absolute reported effect size")
plt.title("Reported effects are noisier in small evaluations")
plt.show()

## Classroom discussion questions

1. If most human evaluations use 100 items and 3 raters/item, what effect sizes can they detect?
2. What is cheaper in your setting: more items, or more raters per item?
3. What information should papers report so that readers can judge statistical power?
